# Tuần 10–11 — Train keypoint sân (32 điểm, YOLO-pose)
Dữ liệu = Roboflow *football-field-detection* + **nhãn tự sinh từ SoccerNet GSR** (cùng domain với dữ liệu đánh giá).
Model có sẵn của Roboflow lệch trung vị 5.8 m trên GSR test; nhãn sinh từ GSR khớp ~0.15 m.

In [ ]:
# Bật GPU (P100 hoặc T4) + Internet trong Settings của notebook
!git clone https://github.com/thaisanh1602/football_cv_tracking.git
%cd football_cv_tracking
!pip install -q -r requirements.txt

## 1. Dataset Roboflow
Dùng lại secret `ROBOFLOW_API_KEY`. Trang dataset *football-field-detection* (workspace `roboflow-jvuqo`) → tab **Dataset** → chọn version **không bị Stretch** → **Download Dataset → YOLOv8 → Show download code** để lấy đúng project + version.

In [ ]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

rf = Roboflow(api_key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY"))
project = rf.workspace("roboflow-jvuqo").project("football-field-detection-f07vi")  # kiểm tra lại slug
project.version(1).download("yolov8", location="/kaggle/working/pitch")  # đổi version cho đúng

In [ ]:
# Ảnh phải giữ tỷ lệ 16:9, không bị stretch thành hình vuông
from collections import Counter
from pathlib import Path
from PIL import Image

for split in ["train", "valid", "test"]:
    imgs = list(Path(f"/kaggle/working/pitch/{split}/images").glob("*"))
    print(split, len(imgs), Counter(Image.open(i).size for i in imgs).most_common(3))

In [ ]:
!python scripts/prepare_pitch_dataset.py --src /kaggle/working/pitch --out /kaggle/working/pitch/data_fixed.yaml

## 2. Nhãn từ SoccerNet GSR (tập train, không trùng clip test dùng để đánh giá)
Tải 30 clip (~6 GB) bằng HTTP range, mỗi clip 1 request. Sinh nhãn mỗi 10 frame → ~2000 ảnh. Phải thấy phần lớn frame được giữ và *khớp kém* ít.

In [ ]:
!python scripts/download_gsr.py --split train --n-clips 30 --out /tmp/gsr/train
!python scripts/make_pitch_labels_gsr.py --gsr-root /tmp/gsr/train --out /kaggle/working/pitch_gsr --stride 10 --val-clips 3
!rm -rf /tmp/gsr  # giải phóng ổ đĩa, ảnh cần dùng đã được copy sang pitch_gsr

In [ ]:
# Gộp 2 nguồn vào 1 data.yaml (Ultralytics nhận danh sách thư mục)
import sys, yaml
sys.path.insert(0, ".")
from src.pitch.config import PITCH

combined = {
    "train": ["/kaggle/working/pitch_gsr/images/train", "/kaggle/working/pitch/train/images"],
    "val": ["/kaggle/working/pitch_gsr/images/val", "/kaggle/working/pitch/valid/images"],
    "kpt_shape": [32, 3],
    "flip_idx": PITCH.flip_idx,
    "names": {0: "pitch"},
}
Path("/kaggle/working/pitch_combined.yaml").write_text(yaml.safe_dump(combined, sort_keys=False))
print(Path("/kaggle/working/pitch_combined.yaml").read_text())

## 3. Train
Chạy thử `--epochs 1` trước, ổn rồi đổi lại và **Save Version → Save & Run All**. ~2400 ảnh ở imgsz 1280: 50 epoch mất vài giờ trên T4/P100 (giới hạn 12 h). Nếu `CUDA out of memory` thì `--batch 4`.

In [ ]:
!python scripts/train_pitch.py --data /kaggle/working/pitch_combined.yaml --model yolo11m-pose.pt --imgsz 1280 --epochs 50 --batch 8 --project /kaggle/working/runs

In [ ]:
!cp /kaggle/working/runs/pitch32/weights/best.pt /kaggle/working/pitch_best.pt

## 4. Đánh giá trên GSR test (so với model Roboflow: trung vị 5.8 m)
Có thể chạy ở đây hoặc ở máy local sau khi tải `pitch_best.pt` về.

In [ ]:
!python scripts/download_gsr.py --split test --clips SNGS-116 SNGS-117 SNGS-118 --out /tmp/gsr/test
!python scripts/eval_pitch.py --data-root /tmp/gsr/test --pitch-weights /kaggle/working/pitch_best.pt --pitch-imgsz 1280 --out /kaggle/working/pitch_eval